# 02 数据清洗、变量构造与合并

本 Notebook 完成：
- 各原始数据表的清洗（编码识别、主键统一、日期处理、类型转换）
- 公司—年度面板的构造
- 13 个核心指标的计算
- 缩尾（Winsorize）处理
- 多表合并与输出

In [ ]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

BASE = "."  # 项目根目录

def read_csmar_excel(path):
    return pd.read_excel(path, header=0, skiprows=[1])

# ============================================================
# 2.1 资产负债表：清洗与列名映射
# ============================================================
BS_COLS = {
    'code': 'code', 'stknme': 'stknme', 'listingDate': 'listingDate', 'EndDate': 'EndDate',
    'FS_Combas-A001101000': 'cash', 'FS_Combas-A001100000': 'current_assets',
    'FS_Combas-A001000000': 'total_assets', 'FS_Combas-A002101000': 'st_loan',
    'FS_Combas-A002125000': 'current_liab', 'FS_Combas-A002201000': 'lt_loan',
    'FS_Combas-A002200000': 'noncurrent_liab', 'FS_Combas-A002000000': 'total_liab',
    'FS_Combas-A003000000': 'equity',
}

def clean_balance_sheet(df):
    df = df.rename(columns={k: v for k, v in BS_COLS.items() if k in df.columns})
    cols = ['code','stknme','listingDate','EndDate','cash','current_assets','total_assets',
            'st_loan','current_liab','lt_loan','noncurrent_liab','total_liab','equity']
    df = df[[c for c in cols if c in df.columns]].copy()
    df['code'] = df['code'].astype(str).str.zfill(6)
    df['EndDate'] = pd.to_datetime(df['EndDate'], errors='coerce')
    df['year'] = df['EndDate'].dt.year
    num_cols = ['cash','current_assets','total_assets','st_loan','current_liab',
                'lt_loan','noncurrent_liab','total_liab','equity']
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df[df['year'] >= 2000].copy()

bs1 = read_csmar_excel("data/raw/balance_sheet_2000_2010/跨表查询_沪深京股票(年频).xlsx")
bs2 = read_csmar_excel("data/raw/balance_sheet_2011_2024/跨表查询_沪深京股票(年频).xlsx")
bs = pd.concat([clean_balance_sheet(bs1), clean_balance_sheet(bs2)])
bs = bs.sort_values('EndDate').drop_duplicates(subset=['code','year'], keep='last')
print(f"资产负债表: {len(bs)} 行, {bs['code'].nunique()} 家公司")
bs.head(3)

## 说明：资产负债表清洗规则

1. **主键统一**：`code` 统一为 6 位字符串（`str.zfill(6)`）
2. **日期处理**：`EndDate` 转为 `datetime64`，提取 `year` 为整数
3. **重复值**：按 `code-year` 去重，保留 `EndDate` 最大的记录（通常为年报）
4. **年份筛选**：保留 2000 年及以后

In [ ]:
# 利润表/现金流量表清洗
IC_COLS = {
    'code': 'code', 'EndDate': 'EndDate',
    'FS_Comins-B001101000': 'revenue',
    'FS_Comins-B002000000': 'net_profit',
    'FS_Comscfd-C001000000': 'cfo',
    'FS_Comscfd-C006000000': 'cash_end',
}

def clean_income(df):
    df = df.rename(columns={k: v for k, v in IC_COLS.items() if k in df.columns})
    cols = ['code','EndDate','revenue','net_profit','cfo','cash_end']
    df = df[[c for c in cols if c in df.columns]].copy()
    df['code'] = df['code'].astype(str).str.zfill(6)
    df['EndDate'] = pd.to_datetime(df['EndDate'], errors='coerce')
    df['year'] = df['EndDate'].dt.year
    for c in ['revenue','net_profit','cfo','cash_end']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df[df['year'] >= 2000].copy()

ic1 = read_csmar_excel("data/raw/income_cashflow_2000_2010/跨表查询_沪深京股票(年频).xlsx")
ic2 = read_csmar_excel("data/raw/income_cashflow_2011_2024/跨表查询_沪深京股票(年频).xlsx")
ic = pd.concat([clean_income(ic1), clean_income(ic2)])
ic = ic.sort_values('EndDate').drop_duplicates(subset=['code','year'], keep='last')
print(f"利润表/现金流量表: {len(ic)} 行")

In [ ]:
# 股权结构数据清洗
common = read_csmar_excel("data/raw/csmar_common_vars/常用变量查询（年度）.xlsx")
own = common[['Stkcd','accper','Shrcr1','Shrhfd5']].copy()
own = own.dropna(subset=['Stkcd','accper'])
own['code'] = own['Stkcd'].astype(int).astype(str).str.zfill(6)
own['year'] = own['accper'].astype(int)
own['Top1'] = pd.to_numeric(own['Shrcr1'], errors='coerce') / 100   # 百分比转小数
own['HHI5'] = pd.to_numeric(own['Shrhfd5'], errors='coerce')         # 已为小数
own = own[['code','year','Top1','HHI5']].drop_duplicates(subset=['code','year'], keep='last')
print(f"股权数据: {len(own)} 行, Top1均值: {own['Top1'].mean():.3f}")

In [ ]:
# 公司基本信息清洗
firm = read_csmar_excel("data/raw/firm_info_annual/STK_LISTEDCOINFOANL.xlsx")
firm['code'] = firm['Symbol'].astype(str).str.zfill(6)
firm['EndDate'] = pd.to_datetime(firm['EndDate'], errors='coerce')
firm['year'] = firm['EndDate'].dt.year
firm['list_date'] = pd.to_datetime(firm['LISTINGDATE'], errors='coerce')
firm['list_year'] = firm['list_date'].dt.year
firm['industry_code'] = firm['IndustryCode'].astype(str).str[0]
industry_map = {
    'A':'农林牧渔业','B':'采矿业','C':'制造业','D':'电力/热力/燃气/水业',
    'E':'建筑业','F':'批发零售业','G':'交通运输/仓储/邮政业',
    'H':'住宿餐饮业','I':'信息技术服务业','J':'金融业','K':'房地产业',
    'L':'租赁商务服务业','M':'科学研究技术服务业','N':'水利/环境/公共设施管理业',
    'O':'居民服务业','P':'教育','Q':'卫生社会工作','R':'文化体育娱乐业','S':'综合'
}
firm['industry_name'] = firm['industry_code'].map(industry_map).fillna('其他')
firm_info = firm[['code','year','ShortName','industry_code','industry_name',
                   'list_year','LISTINGSTATE']].drop_duplicates(subset=['code','year'], keep='last')
print(f"公司信息: {len(firm_info)} 行, {firm_info['code'].nunique()} 家公司")

## 2.5 合并面板数据

以 `code-year` 为主键，依次左连接各数据表。每次合并前后记录行数变化。

In [ ]:
# 多表合并
print(f"步骤1 资产负债表: {len(bs)} 行")
panel = bs.merge(ic[['code','year','revenue','net_profit','cfo']], on=['code','year'], how='left')
print(f"步骤2 + 利润表: {len(panel)} 行")
panel = panel.merge(own, on=['code','year'], how='left')
print(f"步骤3 + 股权结构: {len(panel)} 行, Top1非缺失: {panel['Top1'].notna().sum()}")
panel = panel.merge(firm_info, on=['code','year'], how='left')
print(f"步骤4 + 公司信息: {len(panel)} 行")

# 补充list_year（部分年份可能缺失）
ly_map = firm_info.groupby('code')['list_year'].min()
panel['list_year'] = panel['list_year'].fillna(panel['code'].map(ly_map))
panel = panel[panel['total_assets'] > 0].copy()
print(f"步骤5 删除total_assets<=0后: {len(panel)} 行")

## 2.3 指标构造

计算 13 个核心变量。比率变量均为小数形式（如 0.35 表示 35%）。

In [ ]:
# 构造指标变量
panel['Lev_raw']   = panel['total_liab'] / panel['total_assets']
panel['SL_raw']    = panel['current_liab'] / panel['total_assets']
panel['LL_raw']    = panel['noncurrent_liab'] / panel['total_assets']
panel['SDR_raw']   = np.where(panel['total_liab'] > 0, panel['current_liab']/panel['total_liab'], np.nan)
panel['Cash_raw']  = panel['cash'] / panel['total_assets']
panel['ROA_raw']   = panel['net_profit'] / panel['total_assets']
panel['ROE_raw']   = np.where(panel['equity'] != 0, panel['net_profit']/panel['equity'], np.nan)
panel['SLoan_raw'] = panel['st_loan'] / panel['total_assets']
panel['LLoan_raw'] = panel['lt_loan'] / panel['total_assets']
panel['HHI5_raw']  = panel['HHI5'].copy()
panel['Top1_raw']  = panel['Top1'].copy()
panel['Size']      = np.log(panel['total_assets'])
panel['Age']       = panel['year'] - panel['list_year'] + 1

print("指标构造完成")
panel[['Lev_raw','SL_raw','LL_raw','SDR_raw','Cash_raw','ROA_raw','ROE_raw']].describe()

## 2.4 缩尾处理（Winsorize）

对比率类指标按**年度** 1%-99% 分位数进行缩尾，同时保留原始变量（后缀 `_raw`）。

**原理**：同一指标在不同年份的分布可能有所不同（如金融危机年份极端值更多），按年度缩尾比全样本缩尾更能保留时序变化信息，同时有效控制极端值的影响。

In [ ]:
def winsorize_by_year(df, col, lower=0.01, upper=0.99):
    """按年度在lower-upper分位数缩尾"""
    result = df[col].copy()
    for yr, grp in df.groupby('year'):
        lo = grp[col].quantile(lower)
        hi = grp[col].quantile(upper)
        mask = df['year'] == yr
        result.loc[mask] = result.loc[mask].clip(lo, hi)
    return result

winsor_vars = ['Lev','SL','LL','SDR','Cash','ROA','ROE','SLoan','LLoan','HHI5']
for var in winsor_vars:
    raw = f'{var}_raw'
    if raw in panel.columns:
        panel[var] = winsorize_by_year(panel, raw)

# 缩尾效果比较
for var in winsor_vars:
    raw = f'{var}_raw'
    if raw in panel.columns and var in panel.columns:
        print(f"{var}: 原始均值={panel[raw].mean():.4f} std={panel[raw].std():.4f} | "
              f"缩尾后均值={panel[var].mean():.4f} std={panel[var].std():.4f}")

In [ ]:
# 保存最终数据
panel.to_csv('data/combined/csmar_firm_year_panel.csv', index=False, encoding='utf-8-sig')
panel.to_parquet('data/combined/csmar_firm_year_panel.parquet', index=False)
print(f"已保存: {len(panel)} 行, {panel['code'].nunique()} 家公司, {panel['year'].min()}-{panel['year'].max()}年")
print("data/combined/csmar_firm_year_panel.csv")
print("data/combined/csmar_firm_year_panel.parquet")

## 3.2 进阶存储：Parquet 格式比较

Parquet 是列式存储格式，在读取特定列或大规模数据时有显著优势。

In [ ]:
import time, os
import pyarrow.parquet as pq

# 只读取部分列（列式存储的优势）
df_small = pd.read_parquet('data/combined/csmar_firm_year_panel.parquet',
                            columns=['code','year','Lev','ROA','Cash'])
print(f"仅读5列: {df_small.shape}")

schema = pq.read_schema('data/combined/csmar_firm_year_panel.parquet')
print(f"Schema字段数: {len(schema)}")

t0 = time.time()
pd.read_csv('data/combined/csmar_firm_year_panel.csv')
csv_time = time.time() - t0
csv_size = os.path.getsize('data/combined/csmar_firm_year_panel.csv') / 1024

t0 = time.time()
pd.read_parquet('data/combined/csmar_firm_year_panel.parquet')
pq_time = time.time() - t0
pq_size = os.path.getsize('data/combined/csmar_firm_year_panel.parquet') / 1024

print(f"CSV:     读取 {csv_time:.3f}s, 大小 {csv_size:.1f} KB")
print(f"Parquet: 读取 {pq_time:.3f}s, 大小 {pq_size:.1f} KB")
print(f"Parquet体积/CSV体积 = {pq_size/csv_size:.2f}")